# 07 · Özgün model: Optuna hiperparametre araması

**Önce GPU açın** (T4). Özgün model (Orig_full: tüm bileşenler) için 25 deneme.

- Arama uzayı: diğer modellerle aynı + özgün kaybın `ordinal_sigma` ve `ordinal_lambda` parametreleri (ordinal kayıp etiket yumuşatma kullanmadığından `label_smoothing` aranmaz)
- İlk deneme: aynı omurganın (ConvNeXt-Tiny) en iyi parametreleri (adil başlangıç noktası)
- Aynı bütçe: eğitim kümesinin %40'ı, 3+10 epoch, MedianPruner; amaç validation macro F1
- Süre yaklaşık 1-1,5 saat. Oturum kopsa aynı notebook'u yeniden çalıştırın, kaldığı yerden devam eder
- Sonunda en iyi parametrelerle tam eğitim kümesinde seed 42 ile yeniden eğitilir (`Orig_full__opt`)
- Test kümesi yüklenmez

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
# Kod ve çıktılar Drive'da tek klasörde durur; ham veri oturum diskinde (/content/data) tutulur.
PROJECT = next((p for p in ('/content/drive/MyDrive/YZProje', '/content/drive/MyDrive/YZProje/YZProje')
                if os.path.isfile(p + '/src/config.py')), None)
assert PROJECT, "Drive'da YZProje/src/config.py bulunamadı: proje klasörünü MyDrive/YZProje olarak yerleştirin."
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = PROJECT + '/outputs'
os.environ['KNEE_DATA_ROOT'] = '/content/data'
print('Proje klasörü:', PROJECT)

In [ ]:
!pip install -q imagehash==4.3.2 tabulate==0.9.0 timm==1.0.30 optuna==5.0.0
import torch
assert torch.cuda.is_available(), 'GPU yok! Çalışma zamanı > T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Görüntüler oturum diskinde yoksa: önce Drive'daki arşiv kopyası, o da yoksa bağlantıdan indirip arşivi Drive'a kaydeder.
from src import config as C, data_prep
print('Veri hazır:', data_prep.ensure_data())

## Arama

In [ ]:
from src import optimize as O, original as OR
warm = {k: v for k, v in OR.convnext_hparams().items() if k != 'label_smoothing'}
warm.update(ordinal_sigma=0.75, ordinal_lambda=0.3)
spec = OR.VARIANTS['Orig_full']
study = O.run_search('Original', n_trials=25, subset_fraction=0.4, loss=spec['loss'],
                     model_kwargs=spec['model'], warm_start=warm)
print(study.best_params, study.best_value)

## En iyi parametrelerle tam eğitim (seed 42)

In [ ]:
r = O.retrain_best('Original', seed=42, name='Orig_full__opt', loss=spec['loss'], model_kwargs=spec['model'])
r['val_metrics']

`outputs/results.csv`, `outputs/optuna/Original/search_summary.json` ve `outputs/optuna/Original/trials.csv` dosyalarını paylaşın.